# SafetyVision: Helmet Detection using YOLOv8

Is notebook mein hum Roboflow ka helmet dataset lekar YOLOv8 ko train karte hain, evaluate karte hain aur `best.pt` download karte hain.

**Pehle:** Runtime → Change runtime type → **T4 GPU** select karo.

## 1. Setup

In [ ]:
!pip install -q ultralytics roboflow
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

## 2. Dataset download (Roboflow)

Roboflow Universe par helmet detection dataset kholo → **Download Dataset** → format **YOLOv8** → **show download code** → jo snippet mile use neeche paste karo.

> API key kabhi GitHub par upload mat karna. Notebook upload karne se pehle `YOUR_API_KEY` hi rehne dena.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("WORKSPACE").project("PROJECT")
dataset = project.version(1).download("yolov8")   # version number apne dataset ka daalo
print("Dataset location:", dataset.location)

## 3. Dataset ko samjho (EDA)

In [ ]:
import yaml, glob, os
from collections import Counter

with open(f"{dataset.location}/data.yaml") as f:
    cfg = yaml.safe_load(f)
names = cfg["names"]
print("Classes:", names)

for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{dataset.location}/{split}/images/*")
    print(f"{split:6s}: {len(imgs)} images")

# class distribution (train)
counts = Counter()
for lf in glob.glob(f"{dataset.location}/train/labels/*.txt"):
    for line in open(lf):
        if line.strip():
            counts[int(line.split()[0])] += 1
names_list = names if isinstance(names, list) else [names[i] for i in sorted(names)]
for k, v in sorted(counts.items()):
    print(f"{names_list[k]:20s}: {v} boxes")

In [ ]:
import matplotlib.pyplot as plt
plt.bar([names_list[k] for k in sorted(counts)], [counts[k] for k in sorted(counts)])
plt.title("Class distribution (train)"); plt.ylabel("Boxes"); plt.show()

## 4. Training

`yolov8n.pt` chhota aur fast model hai (Colab free GPU par ~20-40 min). Agar accuracy kam lage to `yolov8s.pt` try karo.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # pretrained COCO weights se start (transfer learning)
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,        # 15 epoch tak improvement na ho to ruk jao
    name="safetyvision",
)
save_dir = str(model.trainer.save_dir)
print("Results saved in:", save_dir)

## 5. Evaluation (validation set)

In [ ]:
best = YOLO(f"{save_dir}/weights/best.pt")
m = best.val()
print(f"mAP@50      : {m.box.map50:.3f}")
print(f"mAP@50-95   : {m.box.map:.3f}")
print(f"Precision   : {m.box.mp:.3f}")
print(f"Recall      : {m.box.mr:.3f}")

In [ ]:
from IPython.display import Image, display
for f in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    p = f"{save_dir}/{f}"
    if os.path.exists(p):
        display(Image(p, width=800))

## 6. Test images par prediction

In [ ]:
pred = best.predict(source=f"{dataset.location}/test/images", conf=0.4, save=True, name="predictions")
pred_dir = str(pred[0].save_dir)
for p in sorted(glob.glob(f"{pred_dir}/*.jpg"))[:6]:
    display(Image(p, width=500))

## 7. Download karo

`best.pt` ko repo ke `weights/` folder mein, aur graphs/predictions ko `results/` folder mein rakhna.

In [ ]:
import shutil
from google.colab import files

os.makedirs("export", exist_ok=True)
shutil.copy(f"{save_dir}/weights/best.pt", "export/best.pt")
for f in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    if os.path.exists(f"{save_dir}/{f}"):
        shutil.copy(f"{save_dir}/{f}", "export/")
for p in sorted(glob.glob(f"{pred_dir}/*.jpg"))[:6]:
    shutil.copy(p, "export/")
shutil.make_archive("safetyvision_export", "zip", "export")
files.download("safetyvision_export.zip")